# Investigating forecast failures

> **Historical v1 analysis.** The modelling target in this workflow ends in September 2025; newer supplemental observations remain separate here. For the current 519-game forecast, see [Notebook 20](20_fixed_cohort_2027_forecast.ipynb). [Notebook 17](17_fixed_sources_integration.ipynb) begins the separate v2 workflow.

This notebook reviews bias, difficult origins and unusual monthly changes before any model selection or additional feature engineering. Gradient boosting wins Indie in three cutoffs, but its 2022-origin error is 31.7% worse than last value. RPG and Strategy illustrate another concern: three fold wins can coexist with worse overall MAE.

## 1. Check stability across cutoffs

Mean error across folds and the number of wins answer different questions. Compare both with the worst fold.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
stability = pd.read_csv(ROOT / "reports/model_diagnostics/stability.csv")
stability

category,model,mean_fold_mae,worst_fold_mae,best_fold_mae,wins_vs_last_value,folds
Action,Damped seasonal,378208.724692,866933.838641,135443.914468,2,4
Action,Damped trend,280759.172211,445097.499309,142242.667820,2,4
Action,Gradient boosting,341764.521367,490602.711426,110006.360496,2,4
Action,Last value,259766.228125,453684.255833,100199.311667,0,4
Action,SES,259766.228945,453684.257037,100199.311657,2,4
Action,Seasonal naive,332754.300625,474038.367500,219853.865833,1,4
Indie,Damped seasonal,59388.526629,94286.506204,36608.577399,3,4
Indie,Damped trend,72435.171474,98368.050607,52092.033807,1,4
Indie,Gradient boosting,54346.871962,63895.056778,44887.378057,3,4
Indie,Last value,66931.965833,95312.465833,42817.899167,0,4


## 2. Inspect overprediction and underprediction

Positive mean error means underprediction, using actual minus predicted.

In [2]:
bias = pd.read_csv(ROOT / "reports/model_diagnostics/bias.csv")
bias.loc[bias.lead_window.eq("Leads 16–27")]

category,model,lead_window,mae,mean_error,underprediction_pct,overprediction_pct,scored_rows
Action,Damped seasonal,Leads 16–27,378208.724692,189057.819199,60.416667,39.583333,48
Action,Damped trend,Leads 16–27,280759.172211,-105136.175600,41.666667,58.333333,48
Action,Gradient boosting,Leads 16–27,341764.521367,297178.951581,81.250000,18.750000,48
Action,Last value,Leads 16–27,259766.228125,157058.095208,60.416667,39.583333,48
Action,SES,Leads 16–27,259766.228945,157058.093650,60.416667,39.583333,48
Action,Seasonal naive,Leads 16–27,332754.300625,245371.771042,81.250000,18.750000,48
Indie,Damped seasonal,Leads 16–27,59388.526629,-43143.235932,25.000000,75.000000,48
Indie,Damped trend,Leads 16–27,72435.171474,-48387.409953,22.916667,77.083333,48
Indie,Gradient boosting,Leads 16–27,54346.871962,26595.993561,70.833333,29.166667,48
Indie,Last value,Leads 16–27,66931.965833,47910.804167,72.916667,27.083333,48


## 3. Inspect unusual activity changes

These six flags are screening candidates, with thresholds based on prior observations. They are not confirmed data errors.

In [3]:
flags = pd.read_csv(ROOT / "reports/model_diagnostics/activity_flags.csv")
flags.loc[flags.unusual_change.eq(True), ["category", "month", "monthly_change_pct", "change_robust_z", "coverage_change_pp", "observed_app_change"]]

category,month,monthly_change_pct,change_robust_z,coverage_change_pp,observed_app_change
Massively Multiplayer,2015-06-01,49.240665,4.056011,0.000000,2.0
Massively Multiplayer,2021-01-01,27.105353,3.655119,-0.802841,-3.0
Massively Multiplayer,2022-12-01,15.935391,4.255566,1.185771,3.0
RPG,2015-11-01,65.330662,3.913612,0.398630,19.0
Strategy,2019-05-01,29.744980,4.041445,-0.140189,-6.0
Strategy,2020-04-01,31.825512,3.758407,-0.284991,-6.0


## 4. Read the largest ML misses

Coverage and movement diagnostics help identify cases for investigation; coincidence alone does not establish a cause.

In [4]:
largest = pd.read_csv(ROOT / "reports/model_diagnostics/largest_errors.csv")
largest.loc[largest.model.eq("Gradient boosting"), ["category", "origin", "month", "lead", "actual", "prediction", "error", "unusual_change", "coverage_change_pp"]]

category,origin,month,lead,actual,prediction,error,unusual_change,coverage_change_pp
Action,2021-06-01,2023-07-01,25,2659009.82,1.895585e+06,763424.396775,False,1.427616
Action,2021-06-01,2023-06-01,24,2636418.66,1.891027e+06,745391.309648,False,-1.182560
Action,2021-06-01,2023-05-01,23,2632688.48,1.956089e+06,676599.392513,False,0.202162
Indie,2020-06-01,2021-01-01,7,978959.52,7.842521e+05,194707.391692,False,-1.455270
RPG,2020-06-01,2022-03-01,21,626619.67,8.094646e+05,-182844.976397,False,-0.553546
Indie,2022-06-01,2023-01-01,7,981373.10,8.025475e+05,178825.635986,False,0.428618
Indie,2022-06-01,2023-07-01,13,941375.51,7.676635e+05,173711.974978,False,0.888383
Massively Multiplayer,2023-06-01,2025-03-01,21,835301.08,6.777971e+05,157504.003422,False,-0.413223
Massively Multiplayer,2022-06-01,2024-08-01,26,806996.56,6.565412e+05,150455.321782,False,0.819672
RPG,2020-06-01,2021-10-01,16,641252.18,7.915193e+05,-150267.123900,False,0.361991


## Diagnostic findings

Model selection was left open at this stage. These diagnostics motivated the following feature experiment against the unchanged gradient-boosting reference. False-alarm and missed-event rates require an event definition and ground truth, so they are not inferred from regression errors or screening flags.

Recorded evidence: [diagnostic manifest](../reports/model_diagnostics/manifest.json).